# Libraries

In [1]:
import torch
import random
import numpy as np

from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
from huggingface_hub import login
from huggingface_hub import create_repo, upload_file

# Global Variables

In [ ]:
MODEL_ID = "EdgeCompress01/Qwen3-4B-WANDA"
MODEL_DIR = "/kaggle/working/model"
REPO_ID = "EdgeCompress01/Qwen3-4B-WANDA-GGUF"
model_name_in_repo = "Sparsity/50/Qwen3-4B-WANDA-Q4_K_M.gguf"
SEED = 42

# Functions

In [3]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Set Seed

In [4]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface Login

In [5]:
# --- 3. HUGGING FACE AUTHENTICATION ---

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Loged in")

Loged in


# Load Model

In [ ]:
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    allow_patterns=["Models/50/*"], # This isolates the download to your specific folder
)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

'/kaggle/working/model'

**Clone llama.cpp**

In [7]:
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 85097, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 85097 (delta 108), reused 40 (delta 40), pack-reused 84917 (from 4)
Receiving objects: 100% (85097/85097), 334.18 MiB | 40.96 MiB/s, done.
Resolving deltas: 100% (61346/61346), done.
/kaggle/working/llama.cpp


**Build llama.cpp**

In [8]:
!cmake -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

# Quantization

**Convert from huggingface to gguf**

In [ ]:
!python convert_hf_to_gguf.py ../model/Models/50 \
    --outfile ../model/llama-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: 25
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch

**Free space**

In [ ]:
!rm -rf ../model/Models/50/*.safetensors
!rm -rf ../model/Models/50/tokenizer*
!rm -rf ../model/Models/50/config*

**Quantize**

In [16]:
!./build/bin/llama-quantize \
    ../model/llama-f16.gguf \
    ../model/llama-q4.gguf \
    q4_k_m

main: build = 8475 (49bfddeca)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '../model/llama-f16.gguf' to '../model/llama-q4.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 31 key-value pairs and 255 tensors from ../model/llama-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_p f32              = 0.900000
llama_model_loader: - kv   3:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   4:                               general.name str              = 25
llama_model_loader: - kv   5:                            general.version str              = 25
llama_model_loader: - kv

**Free space**

In [17]:
!rm ../model/llama-f16.gguf

# PUSH TO HUGGING FACE

In [18]:
# Create repo
create_repo(REPO_ID, repo_type="model", exist_ok=True)

# Upload quantized model
upload_file(
    path_or_fileobj=f"{MODEL_DIR}/llama-q4.gguf",
    path_in_repo=model_name_in_repo,
    repo_id=REPO_ID,
    repo_type="model"
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!
